In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
gro_df = spark.read.format('csv').option('inferSchema', 'true').option('header', 'true').load("data/datasets/groceries.csv")

In [4]:
gro_df.show(10)

+--------+--------+--------+----------+--------+
|order_id|location|    item|order_date|quantity|
+--------+--------+--------+----------+--------+
|      o1| Seattle| Bananas|01/01/2017|       7|
|      o2|    Kent|  Apples|02/01/2017|      20|
|      o3|Bellevue| Flowers|02/01/2017|      10|
|      o4| Redmond|    Meat|03/01/2017|      40|
|      o5| Seattle|Potatoes|04/01/2017|       9|
|      o6|Bellevue|   Bread|04/01/2017|       5|
|      o7| Redmond|   Bread|05/01/2017|       5|
|      o8|Issaquah|   Onion|05/01/2017|       4|
|      o9| Redmond|  Cheese|05/01/2017|      15|
|     o10|Issaquah|   Onion|06/01/2017|       4|
+--------+--------+--------+----------+--------+
only showing top 10 rows



In [5]:
gro_df.createOrReplaceTempView("grocery_view")

In [6]:
# spark.sql("DROP DATABASE IF EXISTS big_data_025320") # -  Works only if no tables in the database. Else Gives error
## AnalysisException: org.apache.hadoop.hive.metastore.api.AlreadyExistsException: Database big_data_025320 already exists
# spark.sql("DROP DATABASE IF EXISTS big_data_025320 CASCADE") - Deletes the tables in database as well, if any!

In [7]:
spark.sql("CREATE DATABASE big_data_025320")

""


In [8]:
spark.sql("show tables").show(5)

+--------+--------------+-----------+
|database|     tableName|isTemporary|
+--------+--------------+-----------+
| default|         1htab|      false|
| default|41group_movies|      false|
| default| 4group_movies|      false|
| default|          4tab|      false|
| default| 6_flags_simon|      false|
+--------+--------------+-----------+
only showing top 5 rows



In [9]:
spark.sql("use big_data_025320")

""


In [10]:
spark.sql("show tables").show(5)

+--------+------------+-----------+
|database|   tableName|isTemporary|
+--------+------------+-----------+
|        |grocery_view|       true|
+--------+------------+-----------+



In [11]:
spark.sql("describe extended grocery_view")

col_name,data_type,comment
order_id,string,null
location,string,null
item,string,null
order_date,string,null
quantity,int,null


In [12]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS big_data_025320.grocery
    (order_id string, location string, item string, order_date timestamp, quantity string)
    USING PARQUET
    AS SELECT order_id, location, item, 
        to_timestamp(order_date) as order_date, quantity
    FROM grocery_view
""")

ParseException: 
Operation not allowed: Schema may not be specified in a Create Table As Select (CTAS) statement(line 2, pos 4)

== SQL ==

    CREATE TABLE IF NOT EXISTS big_data_025320.grocery
----^^^
    (order_id string, location string, item string, order_date timestamp, quantity string)
    USING PARQUET
    AS SELECT order_id, location, item, 
        to_timestamp(order_date) as order_date, quantity
    FROM grocery_view


    CREATE TABLE IF NOT EXISTS big_data_025320.grocery
    (order_id string, location string, item string, order_date timestamp, quantity string)
    
Defining schema here in CREATE statement, which doesn't work with CTAS statement. Which is exactly what the error says:
#### ParseException: 
#### Operation not allowed: Schema may not be specified in a Create Table As Select (CTAS) statement(line 2, pos 4)

## So 2 different approaches/methods:
### Method 1: The CTAS Method
The Create Table As Select (CTAS) syntax is the most concise. You don't define the columns in parentheses; instead, Spark looks at the names and types generated by your SELECT clause.

```
spark.sql("""
    CREATE TABLE IF NOT EXISTS big_data_025320.grocery
    USING PARQUET
    AS 
    SELECT 
        order_id, 
        location, 
        item, 
        to_timestamp(order_date) as order_date, 
        quantity
    FROM grocery_view
""")
```

### Method 2: Explicit Schema + Insert
##### 1. Create the empty Managed Table
```
spark.sql("""
    CREATE TABLE IF NOT EXISTS big_data_025320.grocery
    (order_id string, location string, item string, order_date timestamp, quantity string)
    USING PARQUET
""")
```
#### 2. Insert the data from the view
```
spark.sql("""
    INSERT INTO big_data_025320.grocery
    SELECT order_id, location, item, to_timestamp(order_date), quantity
    FROM grocery_view
""")
```

### Let's try each approach
#### Method 1: CTAS

In [13]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS big_data_025320.grocery
        USING PARQUET
        AS SELECT order_id, location, item, to_timestamp(order_date) as order_date, quantity
        FROM grocery_view
""")

""


In [15]:
spark.sql("describe extended big_data_025320.grocery").show(truncate=False)

+----------------------------+---------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                        |comment|
+----------------------------+---------------------------------------------------------------------------------+-------+
|order_id                    |string                                                                           |null   |
|location                    |string                                                                           |null   |
|item                        |string                                                                           |null   |
|order_date                  |timestamp                                                                        |null   |
|quantity                    |int                                                                              |null   |
|                            |  

In [16]:
spark.sql("DROP TABLE big_data_025320.grocery")

""


#### Method 2: Creating and empty table with schema defined and then inserting the data into it from View

In [17]:
spark.sql("""
        CREATE TABLE IF NOT EXISTS big_data_025320.grocery 
        (order_id string, location string, item string, order_date timestamp, quantity int)
        USING PARQUET
    """)

""


In [18]:
spark.sql("show tables")

database,tableName,isTemporary
big_data_025320,grocery,false
,grocery_view,true


In [19]:
spark.sql("""
    INSERT INTO big_data_025320.grocery
    SELECT order_id, location, item, to_timestamp(order_date) as order_date, quantity
    FROM grocery_view
""")

""


In [22]:
spark.sql("select * from big_data_025320.grocery limit 10").show()

+--------+--------+--------+----------+--------+
|order_id|location|    item|order_date|quantity|
+--------+--------+--------+----------+--------+
|      o1| Seattle| Bananas|      null|       7|
|      o2|    Kent|  Apples|      null|      20|
|      o3|Bellevue| Flowers|      null|      10|
|      o4| Redmond|    Meat|      null|      40|
|      o5| Seattle|Potatoes|      null|       9|
|      o6|Bellevue|   Bread|      null|       5|
|      o7| Redmond|   Bread|      null|       5|
|      o8|Issaquah|   Onion|      null|       4|
|      o9| Redmond|  Cheese|      null|      15|
|     o10|Issaquah|   Onion|      null|       4|
+--------+--------+--------+----------+--------+



In [24]:
spark.sql("describe extended big_data_025320.grocery").show(truncate=False)

+----------------------------+---------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                        |comment|
+----------------------------+---------------------------------------------------------------------------------+-------+
|order_id                    |string                                                                           |null   |
|location                    |string                                                                           |null   |
|item                        |string                                                                           |null   |
|order_date                  |timestamp                                                                        |null   |
|quantity                    |int                                                                              |null   |
|                            |  

### In the HDFS a table has been created
```
[user@g02 ~]$ hdfs dfs -ls -h ./warehouse/big_data_025320.db/grocery
Found 2 items
-rw-r--r--   3 user supergroup          0 2026-03-28 00:35 warehouse/big_data_025320.db/grocery/_SUCCESS
-rw-r--r--   3 user supergroup      1.6 K 2026-03-28 00:35 warehouse/big_data_025320.db/grocery/part-00000-c0417f03-4158-4ecb-93c3-ebb5645acc68-c000.snappy.parquet
[user@g02 ~]$ 
```

### Now let's delete the table and see what happens to metadata and table data files

In [27]:
spark.sql("DROP TABLE big_data_250320.grocery")

""


In [28]:
spark.sql("show tables")

database,tableName,isTemporary
,grocery_view,true


In [29]:
spark.sql("describe extended big_data_250320.grocery")

AnalysisException: Table or view not found for 'DESCRIBE TABLE': big_data_250320.grocery; line 1 pos 0;
'DescribeRelation true, [col_name#525, data_type#526, comment#527]
+- 'UnresolvedTableOrView [big_data_250320, grocery], DESCRIBE TABLE, true


#### Checking the database files physically:
```
[user@g02 ~]$ hdfs dfs -ls -h ./warehouse/big_data_025320.db/grocery
ls: `./warehouse/big_data_025320.db/grocery': No such file or directory
[user@g02 ~]$ 
```

#### So both the metadata and data are gone. deleted. For managed table as expected!

In [30]:
spark.stop()